 <style>
.markdown-content {
    direction: rtl;
    text-align: right;
}

.markdown-content pre {
    direction: ltr;
    text-align: left;
}

.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

# Retrieval vs Generation

## ۱. تعریف‌ها در یک نگاه

| | Retrieval | Generation |
|---|---|---|
| **استعاره** | کتابدار (پیدا می‌کنه) | پژوهشگر (می‌نویسه) |
| **سؤالی که جواب می‌ده** | «کدوم تکه‌ها مرتبطن؟» | «با این مدارک جواب چیه؟» |
| **ورودی** | سؤال + فیلترها | prompt (دستور + context + سؤال) |
| **خروجی** | لیست chunk ها با امتیاز | متن جواب |
| **ماهیت** | ریاضی / جستجو | زبانی / استدلالی |
| **سرعت** | میلی‌ثانیه | ثانیه |
| **هزینه** | کم (بدون token) | زیاد (به ازای token) |
| **قطعیت** | بالا (Deterministic) | پایین‌تر (Stochastic) |
| **خطای معمول** | Miss (نیاورد) / Noise (بی‌ربط آورد) | Hallucination / نادیده گرفتن context |
| **معیار ارزیابی** | Precision، Recall، MRR، nDCG | Faithfulness، Answer Relevance |
| **ابزار بهبود** | chunking، embedding، hybrid، rerank | prompt، مدل، temperature |

## ۲. چرا این تفکیک مهمه؟

وقتی کاربر می‌گه «جواب غلط بود»، علت می‌تونه در Retrieval یا Generation باشه و راه‌حل‌ها کاملاً متفاوتن. بدون این تفکیک، تیم‌ها ماه‌ها prompt رو تغییر می‌دن در حالی که مشکل اصلی retrieval بوده.

## ۳. نکات کلیدی

1. **بیشتر خطاهای واقعی RAG از Retrieval میان**، نه LLM. اگه chunk درست نیاد، هیچ LLM ای نمی‌تونه درست جواب بده.
2. **اتصال این دو، context window است.** K کم ← ریسک از دست دادن مدرک. K زیاد ← نویز، هزینه، و پدیده‌ی **Lost in the Middle** (توجه کمتر LLM به وسط context).
3. **همیشه جدا ارزیابی کن**:
   - آیا chunk درست پیدا شد؟ (ارزیابی Retrieval)
   - با فرض داشتن chunk درست، آیا LLM درست جواب داد؟ (ارزیابی Generation)
4. **همیشه جدا لاگ کن** تا موقع خطا بدونی مشکل کجاست.

## ۴. جدول عیب‌یابی

| علامت | علت احتمالی | مشکل در | اقدام |
|---|---|---|---|
| chunk درست اصلاً نیومد | chunking/embedding ضعیف | Retrieval | بهبود chunking، hybrid search |
| chunk درست بود ولی رتبه پایین | رتبه‌بندی ضعیف | Retrieval | reranking، افزایش K |
| chunk درست تو context بود ولی جواب غلط | prompt ضعیف / مدل ضعیف | Generation | بهبود prompt، مدل قوی‌تر |
| جواب از خود LLM ساخته شده | نبود دستور grounding | Generation | «فقط از context جواب بده»، بررسی faithfulness |

## ۵. ایده‌ی کد این نوت‌بوک

- `retrieve` و `generate` دو تابع **مستقل** هستن.
- برای هر درخواست یک **trace ساختاریافته** (با `request_id`، latency و token) لاگ می‌شه.
- **ارزیابی Retrieval**: Generation حذف می‌شه و فقط Hit Rate سنجیده می‌شه.
- **ارزیابی Generation**: Retrieval دور زده می‌شه و chunk درست (gold) مستقیم به LLM داده می‌شه.
- **کنترل دسترسی** از همین ابتدا از طریق فیلتر metadata اعمال می‌شه.
- در این نوت‌بوک به‌جای مدل واقعی، از embedder و LLM ساختگی استفاده شده تا بدون API و نصب پکیج اضافه اجرا بشه. در فازهای بعد با مدل واقعی جایگزین می‌شن.

## ۶. چک‌لیست production

- [ ] `retrieve` و `generate` دو تابع مستقل هستن (قابل تست و جایگزینی جداگانه).
- [ ] برای هر درخواست یک `request_id` و trace ساختاریافته لاگ می‌شه.
- [ ] latency و تعداد token هر مرحله جداگانه ثبت می‌شه.
- [ ] حالت `no_results` صریحاً مدیریت می‌شه (بدون فراخوانی LLM و بدون هزینه).
- [ ] یک مجموعه‌ی تست کوچک برای Retrieval و یک مجموعه برای Generation وجود داره.
- [ ] K و فیلترها قابل تنظیم هستن (hardcode نیستن).

In [1]:
import re
import time
import uuid
import json
import zlib
import logging
from dataclasses import dataclass
from types import SimpleNamespace

import numpy as np

logging.basicConfig(level=logging.INFO, format="%(message)s")
logger = logging.getLogger("rag")


# ======================================================================
# 1) ساختار داده
# ======================================================================
@dataclass
class Hit:
    id: str
    text: str
    score: float
    metadata: dict


# ======================================================================
# 2) Embedder ساختگی (Hashing Bag-of-Words)
#    فقط برای آموزش؛ در فاز ۴ با مدل واقعی جایگزین می‌شه
# ======================================================================
class ToyEmbedder:
    STOPWORDS = {"و", "در", "به", "از", "که", "را", "با", "این", "است", "هر", "می", "ده", "چی", "چطوریه"}

    def __init__(self, dim: int = 256):
        self.dim = dim

    def _tokens(self, text: str):
        return [t for t in re.findall(r"\w+", text.lower()) if t not in self.STOPWORDS]

    def _embed(self, text: str) -> np.ndarray:
        vec = np.zeros(self.dim, dtype=np.float32)
        for tok in self._tokens(text):
            vec[zlib.crc32(tok.encode("utf-8")) % self.dim] += 1.0
        norm = np.linalg.norm(vec)
        return vec / norm if norm > 0 else vec

    def embed_documents(self, texts):
        return [self._embed(t) for t in texts]

    def embed_query(self, text):
        return self._embed(text)


# ======================================================================
# 3) Vector Store درون‌حافظه‌ای با فیلتر metadata
# ======================================================================
class InMemoryStore:
    def __init__(self):
        self.ids, self.texts, self.metas = [], [], []
        self.matrix = None

    def upsert(self, ids, texts, metadatas, vectors):
        self.ids += ids
        self.texts += texts
        self.metas += metadatas
        new = np.vstack(vectors)
        self.matrix = new if self.matrix is None else np.vstack([self.matrix, new])

    def search(self, q_vec, k=5, filters=None):
        filters = filters or {}
        scores = self.matrix @ q_vec            # بردارها نرمال هستن ⇒ cosine similarity
        order = np.argsort(-scores)
        hits = []
        for i in order:
            if scores[i] <= 0:
                break
            meta = self.metas[i]
            if any(meta.get(key) not in allowed for key, allowed in filters.items()):
                continue
            hits.append(Hit(self.ids[i], self.texts[i], float(scores[i]), meta))
            if len(hits) == k:
                break
        return hits


# ======================================================================
# 4) LLM ساختگی (اولین منبع داخل context رو برمی‌گردونه)
# ======================================================================
class FakeLLM:
    def generate(self, prompt: str):
        m = re.search(r"^\[1\]\s*(.+)$", prompt, flags=re.MULTILINE)
        text = f"بر اساس [1]: {m.group(1)}" if m else "اطلاعاتی پیدا نشد."
        usage = SimpleNamespace(
            prompt_tokens=len(prompt.split()),
            output_tokens=len(text.split()),
        )
        return SimpleNamespace(text=text, usage=usage)


# ======================================================================
# 5) داده‌ی نمونه HR و ایندکس‌کردن
# ======================================================================
docs = [
    ("c1", "مرخصی بدون حقوق حداکثر ۳۰ روز در سال است و باید با تأیید مدیر مستقیم و منابع انسانی ثبت شود.",
     {"source": "leave_policy.pdf", "page": 4, "dept": "public"}),
    ("c2", "مرخصی استحقاقی هر کارمند ۲۶ روز در سال است.",
     {"source": "leave_policy.pdf", "page": 2, "dept": "public"}),
    ("c3", "بیمه تکمیلی هزینه‌های درمان، دندانپزشکی و بستری را پوشش می‌دهد.",
     {"source": "insurance.md", "page": 1, "dept": "public"}),
    ("c4", "جدول حقوق و مزایای مدیران ارشد محرمانه است.",
     {"source": "salary_policy.pdf", "page": 9, "dept": "management"}),
]

embedder = ToyEmbedder()
store = InMemoryStore()
store.upsert(
    ids=[d[0] for d in docs],
    texts=[d[1] for d in docs],
    metadatas=[d[2] for d in docs],
    vectors=embedder.embed_documents([d[1] for d in docs]),
)
llm = FakeLLM()


# ======================================================================
# 6) دو مرحله‌ی مستقل: retrieve و generate
# ======================================================================
def build_prompt(context: str, question: str) -> str:
    return (
        "فقط بر اساس متن‌های زیر جواب بده. "
        "اگر جواب در متن‌ها نیست، بگو «اطلاعاتی پیدا نشد».\n\n"
        f"{context}\n\n"
        f"سؤال: {question}"
    )


def retrieve(question, user_ctx, embedder, store, top_k=5):
    """مرحله‌ی Retrieval: مستقل و قابل تست جداگانه."""
    t0 = time.perf_counter()
    q_vec = embedder.embed_query(question)
    hits = store.search(q_vec, k=top_k, filters=user_ctx["filters"])
    latency = round((time.perf_counter() - t0) * 1000, 2)
    return hits, latency


def generate(question, hits, llm):
    """مرحله‌ی Generation: مستقل و قابل تست جداگانه."""
    t0 = time.perf_counter()
    context = "\n\n".join(f"[{i+1}] {h.text}" for i, h in enumerate(hits))
    response = llm.generate(build_prompt(context, question))
    latency = round((time.perf_counter() - t0) * 1000, 2)
    return response, latency


# ======================================================================
# 7) پایپ‌لاین کامل با trace جداگانه برای هر مرحله
# ======================================================================
def answer(question, user_ctx, embedder, store, llm, top_k=5):
    trace = {"request_id": str(uuid.uuid4())[:8], "question": question}

    hits, r_latency = retrieve(question, user_ctx, embedder, store, top_k)
    trace["retrieval"] = {
        "latency_ms": r_latency,
        "num_hits": len(hits),
        "hits": [
            {"id": h.id, "score": round(h.score, 3), "source": h.metadata.get("source")}
            for h in hits
        ],
    }

    if not hits:
        trace["outcome"] = "no_results"
        logger.info(json.dumps(trace, ensure_ascii=False))
        return {"answer": "اطلاعاتی پیدا نشد.", "sources": []}

    response, g_latency = generate(question, hits, llm)
    trace["generation"] = {
        "latency_ms": g_latency,
        "prompt_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.output_tokens,
    }
    trace["outcome"] = "answered"

    logger.info(json.dumps(trace, ensure_ascii=False))
    return {"answer": response.text, "sources": [h.metadata for h in hits]}


# ======================================================================
# 8) ارزیابی جداگانه‌ی Retrieval (Hit Rate)
# ======================================================================
def evaluate_retrieval(test_cases, embedder, store, top_k=5):
    found = 0
    for case in test_cases:
        hits, _ = retrieve(case["question"], {"filters": {}}, embedder, store, top_k)
        if case["expected_source"] in {h.metadata.get("source") for h in hits}:
            found += 1
    return found / len(test_cases)


# ======================================================================
# 9) ارزیابی جداگانه‌ی Generation (با chunk درست تزریق‌شده)
# ======================================================================
def evaluate_generation(test_cases, llm):
    results = []
    for case in test_cases:
        gold = [Hit("gold", case["gold_chunk"], 1.0, {})]
        response, _ = generate(case["question"], gold, llm)
        results.append({
            "question": case["question"],
            "answer": response.text,
            "passed": case["expected_keyword"] in response.text,
        })
    return results


# ======================================================================
# 10) اجرا
# ======================================================================
print("=" * 60, "\nالف) اجرای پایپ‌لاین کامل\n" + "=" * 60)

user_employee = {"filters": {"dept": ["public"]}}               # کارمند عادی
user_manager = {"filters": {"dept": ["public", "management"]}}  # مدیر

r = answer("مرخصی بدون حقوق چطوریه؟", user_employee, embedder, store, llm)
print("\nپاسخ:", r["answer"])
print("منابع:", r["sources"])

print("\n--- کنترل دسترسی: کارمند عادی درباره‌ی حقوق مدیران می‌پرسه ---")
r = answer("جدول حقوق مدیران", user_employee, embedder, store, llm)
print("پاسخ:", r["answer"])

print("\n--- همان سؤال توسط مدیر ---")
r = answer("جدول حقوق مدیران", user_manager, embedder, store, llm)
print("پاسخ:", r["answer"])

print("\n--- سؤال بی‌ربط (حالت no_results) ---")
r = answer("پایتخت فرانسه کجاست؟", user_employee, embedder, store, llm)
print("پاسخ:", r["answer"])

print("\n" + "=" * 60, "\nب) ارزیابی Retrieval به‌تنهایی\n" + "=" * 60)
retrieval_tests = [
    {"question": "مرخصی بدون حقوق چطوریه؟", "expected_source": "leave_policy.pdf"},
    {"question": "بیمه تکمیلی چی پوشش می‌ده؟", "expected_source": "insurance.md"},
    {"question": "مرخصی استحقاقی چند روزه؟", "expected_source": "leave_policy.pdf"},
]
print(f"Retrieval Hit Rate: {evaluate_retrieval(retrieval_tests, embedder, store):.0%}")

print("\n" + "=" * 60, "\nج) ارزیابی Generation به‌تنهایی\n" + "=" * 60)
generation_tests = [
    {
        "question": "مرخصی استحقاقی چند روزه؟",
        "gold_chunk": "مرخصی استحقاقی هر کارمند ۲۶ روز در سال است.",
        "expected_keyword": "۲۶",
    },
    {
        "question": "بیمه تکمیلی چی پوشش می‌ده؟",
        "gold_chunk": "بیمه تکمیلی هزینه‌های درمان، دندانپزشکی و بستری را پوشش می‌دهد.",
        "expected_keyword": "دندانپزشکی",
    },
]
for res in evaluate_generation(generation_tests, llm):
    print(("✅" if res["passed"] else "❌"), res["question"], "→", res["answer"])

{"request_id": "9194f9f9", "question": "مرخصی بدون حقوق چطوریه؟", "retrieval": {"latency_ms": 0.8, "num_hits": 2, "hits": [{"id": "c1", "score": 0.53, "source": "leave_policy.pdf"}, {"id": "c2", "score": 0.204, "source": "leave_policy.pdf"}]}, "generation": {"latency_ms": 0.21, "prompt_tokens": 52, "output_tokens": 23}, "outcome": "answered"}
{"request_id": "e20bac79", "question": "جدول حقوق مدیران", "retrieval": {"latency_ms": 0.22, "num_hits": 1, "hits": [{"id": "c1", "score": 0.132, "source": "leave_policy.pdf"}]}, "generation": {"latency_ms": 0.03, "prompt_tokens": 41, "output_tokens": 23}, "outcome": "answered"}
{"request_id": "a10a603a", "question": "جدول حقوق مدیران", "retrieval": {"latency_ms": 0.07, "num_hits": 2, "hits": [{"id": "c4", "score": 0.707, "source": "salary_policy.pdf"}, {"id": "c1", "score": 0.132, "source": "leave_policy.pdf"}]}, "generation": {"latency_ms": 0.02, "prompt_tokens": 50, "output_tokens": 11}, "outcome": "answered"}
{"request_id": "069dc88f", "questi

الف) اجرای پایپ‌لاین کامل

پاسخ: بر اساس [1]: مرخصی بدون حقوق حداکثر ۳۰ روز در سال است و باید با تأیید مدیر مستقیم و منابع انسانی ثبت شود.
منابع: [{'source': 'leave_policy.pdf', 'page': 4, 'dept': 'public'}, {'source': 'leave_policy.pdf', 'page': 2, 'dept': 'public'}]

--- کنترل دسترسی: کارمند عادی درباره‌ی حقوق مدیران می‌پرسه ---
پاسخ: بر اساس [1]: مرخصی بدون حقوق حداکثر ۳۰ روز در سال است و باید با تأیید مدیر مستقیم و منابع انسانی ثبت شود.

--- همان سؤال توسط مدیر ---
پاسخ: بر اساس [1]: جدول حقوق و مزایای مدیران ارشد محرمانه است.

--- سؤال بی‌ربط (حالت no_results) ---
پاسخ: اطلاعاتی پیدا نشد.

ب) ارزیابی Retrieval به‌تنهایی
Retrieval Hit Rate: 100%

ج) ارزیابی Generation به‌تنهایی
✅ مرخصی استحقاقی چند روزه؟ → بر اساس [1]: مرخصی استحقاقی هر کارمند ۲۶ روز در سال است.
✅ بیمه تکمیلی چی پوشش می‌ده؟ → بر اساس [1]: بیمه تکمیلی هزینه‌های درمان، دندانپزشکی و بستری را پوشش می‌دهد.
